Ten skrypt definiuje agenta opartego na modelu OpenAI do rozwiązywania problemów matematycznych, a następnie wykorzystuje bibliotekę Arize Phoenix do przeprowadzenia ewaluacji jego wydajności. Generuje syntetyczny zbiór zadań matematycznych, uruchamia agenta w celu ich rozwiązania, ocenia poprawność odpowiedzi za pomocą innego modelu LLM i loguje szczegółowe wyniki oraz dane śledzenia do platformy Phoenix w celu analizy.

# Setup

In [ ]:
!uv pip install -qU "arize-phoenix>=8.0.0" openinference-instrumentation-openai-agents openinference-instrumentation-openai
!uv pip install -q openai nest_asyncio openai-agents  openinference-instrumentation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.2/299.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.1/232.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9

Instaluje lub aktualizuje następujące pakiety:

*   `arize-phoenix`: Biblioteka do monitorowania i analizy modeli uczenia maszynowego, wersja 8.0.0 lub nowsza (`>=8.0.0`). Flaga `-U` oznacza aktualizację pakietu jeśli jest już zainstalowany.
*   `openinference-instrumentation-openai-agents`: Narzędzie do monitorowania i śledzenia działania agentów OpenAI w kontekście wnioskowania (inferencji).
*   `openinference-instrumentation-openai`:  Narzędzie do monitorowania i śledzenia działania modeli OpenAI w kontekście wnioskowania.
*   Flaga `-q` oznacza tryb cichy, co ogranicza ilość wyświetlanych informacji podczas instalacji.

Drugie polecenie:

Instaluje następujące pakiety:

*   `openai`: Oficjalna biblioteka Pythona do interakcji z API OpenAI.
*   `nest_asyncio`: Biblioteka umożliwiająca uruchamianie zagnieżdżonych pętli zdarzeń asynchronicznych, co jest przydatne w niektórych scenariuszach związanych z programowaniem asynchronicznym.
*   `openai-agents`:  Biblioteka do tworzenia i zarządzania agentami opartymi na modelach OpenAI.
*   `openinference-instrumentation`: Narzędzie do monitorowania i śledzenia działania modeli w kontekście wnioskowania (inferencji).
*   Flaga `-q` ponownie oznacza tryb cichy.

In [2]:
# Standard library imports
import asyncio
import os
import uuid

# Third-party imports
import nest_asyncio
import pandas as pd
import phoenix as px
from google.colab import userdata
from opentelemetry.trace import StatusCode, format_span_id
from phoenix.evals import OpenAIModel, llm_classify
from phoenix.experiments import run_experiment
from phoenix.otel import register
from phoenix.trace import SpanEvaluations

# First-party/Local application/library imports
from agents import Agent, Runner, function_tool

# Apply nest_asyncio after imports, if necessary at this global level
nest_asyncio.apply()

1.  **Importy bibliotek zewnętrznych:** Są to pakiety, które trzeba zainstalować oddzielnie (np. za pomocą `pip`).
    *   `nest_asyncio`: Rozwiązuje problemy z uruchamianiem zagnieżdżonych pętli zdarzeń asynchronicznych w niektórych środowiskach.
    *   `pandas`: Oferuje narzędzia do analizy i manipulacji danymi, szczególnie przydatne przy pracy z tabelami danych (DataFrames).
    *   `phoenix as px`: Biblioteka służąca do ewaluacji modeli uczenia maszynowego. Alias `px` pozwala na krótsze odwoływanie się do tej biblioteki w kodzie.
    *   `opentelemetry.trace.StatusCode`, `opentelemetry.trace.format_span_id`: Komponenty biblioteki OpenTelemetry, służącej do śledzenia i monitorowania działania aplikacji.
    *   `phoenix.evals.OpenAIModel`, `phoenix.evals.llm_classify`: Moduły z biblioteki Phoenix związane z wykorzystaniem modeli OpenAI oraz klasyfikacją danych za pomocą dużych modeli językowych (LLM).
    *   `phoenix.experiments.run_experiment`: Funkcja do uruchamiania eksperymentów w ramach biblioteki Phoenix.
    *   `phoenix.otel.register`: Funkcja służąca do rejestrowania komponentów OpenTelemetry w systemie śledzenia.
    *   `phoenix.trace.SpanEvaluations`: Klasa reprezentująca zakres śledzenia (span) związany z ewaluacjami.

2.  **Importy własnych/lokalnych modułów:** Odnoszą się do plików i modułów, które zostały stworzone w ramach tego samego projektu.
    *   `agents`: Moduł zawierający definicje klas `Agent`, `Runner` oraz funkcji `function_tool`. Implementuje logikę związaną z agentami działającymi w systemie.

In [ ]:
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"
os.environ["PHOENIX_CLIENT_HEADERS"] = "api_key=" + userdata.get("arize")
os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")

Ten kod ustawia zmienne środowiskowe w systemie operacyjnym. Zmienne środowiskowe są używane do przechowywania konfiguracji i informacji, które mogą być potrzebne programowi podczas działania.

*   `os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"`: Ustawia zmienną środowiskową `PHOENIX_COLLECTOR_ENDPOINT` na adres URL `https://app.phoenix.arize.com`. Ta zmienna określa punkt końcowy, do którego biblioteka Phoenix (do monitorowania modeli) wysyła dane.

*   `os.environ["PHOENIX_CLIENT_HEADERS"] = "api_key=" + userdata.get('arize')`: Ustawia zmienną środowiskową `PHOENIX_CLIENT_HEADERS`. Ta zmienna zawiera nagłówki HTTP, które będą wysyłane wraz z żądaniami do serwera Phoenix. W szczególności dodaje nagłówek `api_key` z wartością pobraną z danych użytkownika Google Colab za pomocą `userdata.get('arize')`.  Oznacza to, że klucz API dla usługi Arize jest przechowywany w danych użytkownika Colab i używany do autoryzacji żądań.

*   `os.environ["OPENAI_API_KEY"] = userdata.get('openaivision')`: Ustawia zmienną środowiskową `OPENAI_API_KEY` na wartość pobraną z danych użytkownika Google Colab za pomocą `userdata.get('openaivision')`. Ta zmienna przechowuje klucz API dla usług OpenAI, umożliwiając programowi dostęp do modeli i funkcji OpenAI.  Nazwa zmiennej w userdata (`openaivision`) sugeruje, że może być to klucz dedykowany do wizji komputerowej (computer vision).

Podsumowując, ten kod konfiguruje środowisko programu, dostarczając niezbędne informacje uwierzytelniające i adresy URL dla biblioteki Phoenix oraz API OpenAI. Klucze API są pobierane z danych użytkownika Google Colab, co jest typowym sposobem przechowywania poufnych informacji w tym środowisku.

In [4]:
tracer_provider = register(
    project_name="openai-agents-cookbook",
    endpoint="https://app.phoenix.arize.com/v1/traces",
    auto_instrument=True,
)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: openai-agents-cookbook
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'api_key': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



Ten kod inicjalizuje i konfiguruje system śledzenia (tracing) przy użyciu biblioteki Phoenix oraz OpenTelemetry.

*   `tracer_provider = register(...)`: Wywołuje funkcję `register` z modułu `phoenix.otel`. Funkcja ta tworzy i zwraca dostawcę śledzenia (tracer provider), który jest centralnym punktem do zarządzania śledzeniem w aplikacji. Wynikowy obiekt jest przypisywany do zmiennej `tracer_provider`.

*   `project_name="openai-agents-cookbook"`: Określa nazwę projektu jako "openai-agents-cookbook". Ta nazwa będzie używana do identyfikacji danych śledzenia pochodzących z tej aplikacji w systemie Arize Phoenix.

*   `endpoint="https://app.phoenix.arize.com/v1/traces"`: Ustawia punkt końcowy (endpoint) dla wysyłania danych śledzenia do usługi Arize Phoenix. Jest to adres URL, na który będą wysyłane informacje o przebiegu operacji w aplikacji.

*   `auto_instrument=True`: Włącza automatyczne instrumentowanie (auto-instrumentation). Oznacza to, że biblioteka Phoenix spróbuje automatycznie śledzić wiele standardowych operacji i funkcji w kodzie, bez konieczności ręcznego dodawania kodu śledzenia.  To upraszcza proces implementacji śledzenia.


In [ ]:
class CFG:
    model = "gpt-4o"

# Agent 1

In [6]:
@function_tool
def solve_equation(equation: str) -> str:
    """Use python to evaluate the math equation, instead of thinking about it yourself.

    Args:
       equation: string which to pass into eval() in python
    """
    return str(eval(equation))

Podsumowując, ta funkcja przyjmuje wyrażenie matematyczne w postaci ciągu znaków, używa funkcji `eval()` do obliczenia jego wartości i zwraca wynik również w postaci ciągu znaków. Dekorator `@function_tool` wskazuje, że funkcja może być wywoływana przez agenta jako narzędzie do rozwiązywania równań. Należy jednak pamiętać, że użycie `eval()` może być niebezpieczne, jeśli ciąg `equation` pochodzi z niezaufanego źródła, ponieważ pozwala na wykonanie dowolnego kodu Python.

In [ ]:
agent = Agent(
    name="Math Solver",
    instructions="You solve math problems by evaluating them with python and returning the result",
    tools=[solve_equation],
)

In [8]:
# sanity check
result = await Runner.run(agent, "what is 15 + 28?")

print(result)

RunResult:
- Last agent: Agent(name="Math Solver", ...)
- Final output (str):
    15 + 28 equals 43.
- 3 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [9]:
# Get the final output
print(result.final_output)

15 + 28 equals 43.


In [10]:
# Get the entire list of messages recorded to generate the final output
print(result.to_input_list())

[{'content': 'what is 15 + 28?', 'role': 'user'}, {'arguments': '{"equation":"15 + 28"}', 'call_id': 'call_FbklgklvAbf42nBDzQvEcWN7', 'name': 'solve_equation', 'type': 'function_call', 'id': 'fc_682c58cfc51c8191bc6db42c9db2196f087fd859838ab122', 'status': 'completed'}, {'call_id': 'call_FbklgklvAbf42nBDzQvEcWN7', 'output': '43', 'type': 'function_call_output'}, {'id': 'msg_682c58d09f248191bed957000c8e6a9d087fd859838ab122', 'content': [{'annotations': [], 'text': '15 + 28 equals 43.', 'type': 'output_text'}], 'role': 'assistant', 'status': 'completed', 'type': 'message'}]


# Eval

Zadanie / ewaluator / dane.

In [11]:
async def solve_math_problem(dataset_row: dict):
    result = await Runner.run(agent, dataset_row.get("question"))
    return {
        "final_output": result.final_output,
        "messages": result.to_input_list(),
    }


dataset_row = {"question": "What is 15 + 28?"}

result = asyncio.run(solve_math_problem(dataset_row))
print(result)

{'final_output': '15 + 28 equals 43.', 'messages': [{'content': 'What is 15 + 28?', 'role': 'user'}, {'arguments': '{"equation":"15 + 28"}', 'call_id': 'call_72kk1klOcWC4UwVZZG6E6ups', 'name': 'solve_equation', 'type': 'function_call', 'id': 'fc_682c58d153808191a5a1fc8ac0e91d4a0b4910ae5ab5b83c', 'status': 'completed'}, {'call_id': 'call_72kk1klOcWC4UwVZZG6E6ups', 'output': '43', 'type': 'function_call_output'}, {'id': 'msg_682c58d22fdc8191a4da6d6c03a4e42c0b4910ae5ab5b83c', 'content': [{'annotations': [], 'text': '15 + 28 equals 43.', 'type': 'output_text'}], 'role': 'assistant', 'status': 'completed', 'type': 'message'}]}


In [ ]:
def correctness_eval(input, output):
    # Template for evaluating math problem solutions
    MATH_EVAL_TEMPLATE = """
    You are evaluating whether a math problem was solved correctly.

    [BEGIN DATA]
    ************
    [Question]: {question}
    ************
    [Response]: {response}
    [END DATA]

    Assess if the answer to the math problem is correct. First work out the correct answer yourself,
    then compare with the provided response. Consider that there may be different ways to express the same answer
    (e.g., "43" vs "The answer is 43" or "5.0" vs "5").

    Your answer must be a single word, either "correct" or "incorrect"
    """

    # Run the evaluation
    rails = ["correct", "incorrect"]
    eval_df = llm_classify(
        data=pd.DataFrame(
            [{"question": input["question"], "response": output["final_output"]}]
        ),
        template=MATH_EVAL_TEMPLATE,
        model=OpenAIModel(model="gpt-4.1"),
        rails=rails,
        provide_explanation=True,
    )
    label = eval_df["label"][0]
    score = 1 if label == "correct" else 0
    return score

# Syntetyczne dane


In [13]:
MATH_GEN_TEMPLATE = """
You are an assistant that generates diverse math problems for testing a math solver agent.
The problems should include:

Basic Operations: Simple addition, subtraction, multiplication, division problems.
Complex Arithmetic: Problems with multiple operations and parentheses following order of operations.
Exponents and Roots: Problems involving powers, square roots, and other nth roots.
Percentages: Problems involving calculating percentages of numbers or finding percentage changes.
Fractions: Problems with addition, subtraction, multiplication, or division of fractions.
Algebra: Simple algebraic expressions that can be evaluated with specific values.
Sequences: Finding sums, products, or averages of number sequences.
Word Problems: Converting word problems into mathematical equations.

Do not include any solutions in your generated problems.

Respond with a list, one math problem per line. Do not include any numbering at the beginning of each line.
Generate 25 diverse math problems. Ensure there are no duplicate problems.
"""

In [ ]:
pd.set_option("display.max_colwidth", 500)

# Initialize the model
model = OpenAIModel(model=CFG.model, max_tokens=1300)

# Generate math problems
resp = model(MATH_GEN_TEMPLATE)

# Create DataFrame
split_response = resp.strip().split("\n")
math_problems_df = pd.DataFrame(split_response, columns=["question"])
print(math_problems_df.head())

                      question
0         Calculate 15 + 28.  
1           What is 72 - 19?  
2           Multiply 8 by 7.  
3          Divide 144 by 12.  
4  Evaluate (3 + 5) * 2 - 4.  


# Eval 1

In [ ]:
unique_id = uuid.uuid4()

dataset_name = "math-questions-" + str(uuid.uuid4())[:5]

# Upload the dataset to Phoenix
dataset = px.Client().upload_dataset(
    dataframe=math_problems_df,
    input_keys=["question"],
    dataset_name=f"math-questions-{unique_id}",
)
print(dataset)

/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


📤 Uploading dataset...
💾 Examples uploaded: https://app.phoenix.arize.com/datasets/RGF0YXNldDoy/examples
🗄️ Dataset version ID: RGF0YXNldFZlcnNpb246Mg==
Dataset(id='RGF0YXNldDoy', version_id='RGF0YXNldFZlcnNpb246Mg==')


In [ ]:
initial_experiment = run_experiment(
    dataset,
    task=solve_math_problem,
    evaluators=[correctness_eval],
    experiment_description="Solve Math Problems",
    experiment_name=f"solve-math-questions-{str(uuid.uuid4())[:5]}",
)

🧪 Experiment started.
📺 View dataset experiments: https://app.phoenix.arize.com/datasets/RGF0YXNldDoy/experiments
🔗 View this experiment: https://app.phoenix.arize.com/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDox


running tasks |          | 0/25 (0.0%) | ⏳ 00:00<? | ?it/s

/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API comp

✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |          | 0/25 (0.0%) | ⏳ 00:00<? | ?it/s

...

/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(



🔗 View this experiment: https://app.phoenix.arize.com/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDox

Experiment Summary (05/20/25 10:27 AM +0000)
--------------------------------------------
| evaluator        |   n |   n_scores |   avg_score |
|:-----------------|----:|-----------:|------------:|
| correctness_eval |  25 |         25 |           1 |

Tasks Summary (05/20/25 10:26 AM +0000)
---------------------------------------
|   n_examples |   n_runs |   n_errors |
|-------------:|---------:|-----------:|
|           25 |       25 |          0 |


# Eval 2

In [17]:
tracer = tracer_provider.get_tracer(__name__)

In [ ]:
def correctness_eval(input, output):
    MATH_EVAL_TEMPLATE = """
    You are evaluating whether a math problem was solved correctly.

    [BEGIN DATA]
    ************
    [Question]: {question}
    ************
    [Response]: {response}
    [END DATA]

    Assess if the answer to the math problem is correct. First work out the correct answer yourself,
    then compare with the provided response. Consider that there may be different ways to express the same answer
    (e.g., "43" vs "The answer is 43" or "5.0" vs "5").

    Your answer must be a single word, either "correct" or "incorrect"
    """

    # Run the evaluation
    rails = ["correct", "incorrect"]
    eval_df = llm_classify(
        data=pd.DataFrame(
            [{"question": input["question"], "response": output["final_output"]}]
        ),
        template=MATH_EVAL_TEMPLATE,
        model=OpenAIModel(model="gpt-4.1"),
        rails=rails,
        provide_explanation=True,
    )

    return eval_df

In [ ]:
async def solve_math_problem(dataset_row: dict):
    with tracer.start_as_current_span(
        name="agent", openinference_span_kind="agent"
    ) as agent_span:
        question = dataset_row.get("question")
        agent_span.set_input(question)
        agent_span.set_status(StatusCode.OK)

        result = await Runner.run(agent, question)
        agent_span.set_output(result.final_output)

        task_result = {
            "final_output": result.final_output,
            "messages": result.to_input_list(),
        }

        # Evaluation span for correctness
        with tracer.start_as_current_span(
            "correctness-evaluator",
            openinference_span_kind="evaluator",
        ) as eval_span:
            evaluation_result = correctness_eval(dataset_row, task_result)
            eval_span.set_attribute("eval.label", evaluation_result["label"][0])
            eval_span.set_attribute(
                "eval.explanation", evaluation_result["explanation"][0]
            )

        # Logging our evaluation
        span_id = format_span_id(eval_span.get_span_context().span_id)
        score = 1 if evaluation_result["label"][0] == "correct" else 0
        eval_data = {
            "span_id": span_id,
            "label": evaluation_result["label"][0],
            "score": score,
            "explanation": evaluation_result["explanation"][0],
        }
        df = pd.DataFrame([eval_data])
        px.Client().log_evaluations(
            SpanEvaluations(
                dataframe=df,
                eval_name="correctness",
            ),
        )

    return task_result


dataset_row = {"question": "What is 15 + 28?"}

result = asyncio.run(solve_math_problem(dataset_row))
print(result)

llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

{'final_output': 'The result of 15 + 28 is 43.', 'messages': [{'content': 'What is 15 + 28?', 'role': 'user'}, {'arguments': '{"equation":"15 + 28"}', 'call_id': 'call_IlwX0wuztrIBFvUHbDY44XoF', 'name': 'solve_equation', 'type': 'function_call', 'id': 'fc_682c59167d808191bcb1f5f7bdc4878d086c85d8a94ae9a3', 'status': 'completed'}, {'call_id': 'call_IlwX0wuztrIBFvUHbDY44XoF', 'output': '43', 'type': 'function_call_output'}, {'id': 'msg_682c5917330081918e0650893705f67d086c85d8a94ae9a3', 'content': [{'annotations': [], 'text': 'The result of 15 + 28 is 43.', 'type': 'output_text'}], 'role': 'assistant', 'status': 'completed', 'type': 'message'}]}


/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


In [ ]:
initial_experiment = run_experiment(
    dataset,
    task=solve_math_problem,
    experiment_description="Solve Math Problems",
    experiment_name=f"solve-math-questions-{str(uuid.uuid4())[:5]}",
)

🧪 Experiment started.
📺 View dataset experiments: https://app.phoenix.arize.com/datasets/RGF0YXNldDoy/experiments
🔗 View this experiment: https://app.phoenix.arize.com/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDoy


running tasks |          | 0/25 (0.0%) | ⏳ 00:00<? | ?it/s

llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

...

llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API comp

llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


✅ Task runs completed.

🔗 View this experiment: https://app.phoenix.arize.com/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDoy

Tasks Summary (05/20/25 10:28 AM +0000)
---------------------------------------
|   n_examples |   n_runs |   n_errors |
|-------------:|---------:|-----------:|
|           25 |       25 |          0 |
